# The grand-average EEG power spectrum

The EEG's own power spectral density, averaged over every trial, channel and
participant: the background the coherence and the transfer functions are measured
against.

The spectra are the ones `scripts/trfs/get_transfer_functions.py` writes for every trial:
a Welch estimate over 5 s Hann windows at 80% overlap (0.2 Hz resolution, up to the
512 Hz Nyquist frequency), one per reference. Each participant's spectrum is the
listening-time-weighted mean of their trials, and participants are then weighted by their
total analysed listening time, so the grand average is the duration-weighted mean over all
trials. The median and percentiles are weighted the same way. The channel mean is the average of the 64 scalp channels' power (the mastoids
some recordings also carry are left out), not the power of the channel average.

## Figures produced

**`eeg_psd_<reference>`** -- the channel-mean grand-average PSD on log-log axes. The
spread across participants (the 10th-90th percentile of each participant's channel-mean
PSD, weighted by listening time) is shaded grey, with the weighted median dashed. Figure 2's five coherence bands can be
shaded for orientation (`SHADE_COHERENCE_BANDS`, off by default).

A few recordings carry far more power than the rest, and an arithmetic mean follows them:
with them in, the mean lies above most participants below a few hertz. `EXCLUDE_OUTLIERS`
(on by default) leaves out participants whose total power has a robust (modified) z-score
above 3.5, and the cell names who is left out. `EXCLUDE_CHANNEL_OUTLIERS` (also on) applies
the same rule within each channel, for single bad electrodes: the pair is left out of that
channel's averages and that participant's channel mean. The weighted median is the typical
participant either way.

**`eeg_psd_channels_<reference>`** -- every channel's duration-weighted average PSD, coloured
by sensor position as in notebook 6, with the mean over channels in bold black.

**`stimulus_to_eeg_psd_<reference>`** -- for fun: the envelope's PSD (in Pa$^{1.2}$/Hz) divided by
each channel's EEG PSD.

**`transfer_function_magnitude_<reference>`** -- the magnitude of the envelope-to-EEG transfer
function per channel, in µV per Pa$^{0.6}$, unnormalised.

**`implied_vs_grand_average_coherence_<reference>`** -- panel (d) of the overview against
figure 2's grand-average coherence spectrum, as an RMS coherence magnitude over channels.

**`spectra_overview_<reference>`** -- four panels: (a) the envelope PSD, (b) the EEG PSD, (c) the
transfer function magnitude, (d) $|H|^2 S_{xx} / S_{yy}$, the squared coherence the pooled fit
implies.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import LogLocator, NullFormatter, NullLocator

ROOT = Path.cwd().parent if Path.cwd().name == "figures" else Path.cwd()
CONFIG = json.loads((ROOT / "src" / "package_data" / "config.json").read_text())
sys.path.insert(0, str(ROOT / "src"))
from trfs.transfer_function import load_trial, resolve_trf_dir

REFERENCE = "avg"                       # "avg" | "linked_mastoids" | "FCz"
WINDOW_SECONDS, OVERLAP = 5.0, 0.8      # the TRF run the spectra are read from
FEATURE = "gammatone_envelope"          # only selects which file layout load_trial expects

TRF_DIR = resolve_trf_dir(CONFIG, REFERENCE, WINDOW_SECONDS, OVERLAP, ROOT)
OUT_DIR = ROOT / "figures"
CACHE_DIR = OUT_DIR / "figure_data" / REFERENCE
CACHE_DIR.mkdir(parents=True, exist_ok=True)
paths = sorted(TRF_DIR.glob("*_-_eeg.npz"))

# --- house style, shared with the other figure notebooks ---------------------
DEFAULT_FONTSIZE = 10
TICK_FONTSIZE = 8
SMALL_FONTSIZE = 6.5
PANEL_LABEL_FONTSIZE = 14
INK, MUTED, HAIRLINE = "#1b1b1a", "#6b6b68", "#d5d5d1"
NULL_FILL = "#b9b9b4"
plt.rcParams.update({
    "font.size": TICK_FONTSIZE, "axes.labelsize": DEFAULT_FONTSIZE,
    "axes.titlesize": DEFAULT_FONTSIZE, "axes.edgecolor": INK, "axes.labelcolor": INK,
    "text.color": INK, "xtick.color": INK, "ytick.color": INK,
    "xtick.labelsize": TICK_FONTSIZE, "ytick.labelsize": TICK_FONTSIZE,
    "axes.linewidth": 0.8, "figure.dpi": 130, "pdf.fonttype": 42, "ps.fonttype": 42,
})


def bare(ax):
    """Recessive chrome: drop the top/right rules."""
    ax.spines[["top", "right"]].set_visible(False)
    return ax


print(f"{len(paths)} trial files in {TRF_DIR.name} under {REFERENCE}")

## Pooling

One pass over the trial files, cached per reference and window. The EEG is stored in
volts, so the PSD is in V$^2$/Hz; it is converted to µV$^2$/Hz for the figure.

In [ ]:
MASTOIDS = ("A1", "A2")

# The envelope's PSD in physical units, for the last figure: each stimulus's digital-to-pressure
# factor k from its calibrated presentation level, as notebook 6 computes and caches it. The
# envelope is compressed (power 0.6), so its PSD scales by k^(2 * 0.6).
import re

from audio_features.factory import Brian2HearsGammatoneCochleogramExtractor

PRESENTATION_PRESSURE_PA = 20e-6 * 10 ** (62.0 / 20)        # 62 dB(A)
ENVELOPE_COMPRESSION = Brian2HearsGammatoneCochleogramExtractor.compression
calibration_cache = OUT_DIR / "figure_data" / "stimulus_pressure_calibration.npz"
if not calibration_cache.exists():
    raise FileNotFoundError(f"{calibration_cache.name} is written by notebook 6; run it first")
with np.load(calibration_cache) as saved:
    # Tube-filtered A-weighted RMS, notebook 6's default (CALIBRATION_TUBE_FILTER = True).
    ENVELOPE_POWER_FACTOR = {str(name): (PRESENTATION_PRESSURE_PA / rms) ** (2 * ENVELOPE_COMPRESSION)
                             for name, rms in zip(saved["stimuli"], saved["rms_a_tube"])}


def pool_psd():
    """Channel-mean PSD per participant (listening-time weighted), and the per-channel grand
    average over participants."""
    per_subject, stimulus, weights, frequencies, channels = {}, {}, {}, None, None
    for count, path in enumerate(paths, 1):
        trial = load_trial(path, FEATURE)
        psd, names = trial.psd_for(REFERENCE)
        if channels is None:
            # The 64 scalp channels. Recordings with mastoid electrodes also carry A1/A2 (66);
            # those without carry only the 64, so every trial is indexed by name.
            frequencies = trial.frequencies
            channels = [name for name in names if name not in MASTOIDS]
        missing = [name for name in channels if name not in names]
        assert not missing, f"{path.stem}: missing {missing}"
        psd = psd[:, [list(names).index(name) for name in channels]]
        weight = trial.analysed_duration
        total = per_subject.get(trial.subject)
        per_subject[trial.subject] = psd * weight if total is None else total + psd * weight
        name = re.search(r"_-_audio\d+-(.+)_-_eeg$", path.stem).group(1)
        envelope = trial.stimulus_psd * ENVELOPE_POWER_FACTOR[name] * weight   # Pa^1.2/Hz
        stimulus[trial.subject] = envelope + stimulus.get(trial.subject, 0.0)
        weights[trial.subject] = weights.get(trial.subject, 0.0) + weight
        if count % 100 == 0 or count == len(paths):
            print(f"  {count}/{len(paths)} trials", flush=True)
    subjects = sorted(per_subject)
    subject_psd = np.array([per_subject[s] / weights[s] for s in subjects])   # (S, F, C)
    subject_stimulus = np.array([stimulus[s] / weights[s] for s in subjects])  # (S, F)
    return (frequencies, channels, subjects, subject_psd, np.array([weights[s] for s in subjects]),
            subject_stimulus)


cache = CACHE_DIR / f"eeg_psd_{WINDOW_SECONDS:g}s.npz"
usable = False
if cache.exists():
    with np.load(cache) as saved:
        usable = saved["n_files"] == len(paths) and "subject_stimulus_psd" in saved.files
if usable:
    with np.load(cache) as saved:
        frequencies, subject_psd = saved["frequencies"], saved["subject_psd"]
        channels, subjects = list(saved["channels"]), list(saved["subjects"])
        subject_seconds = saved["subject_seconds"]
        subject_stimulus_psd = saved["subject_stimulus_psd"]
    print(f"cached: {cache.name}")
else:
    (frequencies, channels, subjects, subject_psd, subject_seconds,
     subject_stimulus_psd) = pool_psd()
    np.savez(cache, frequencies=frequencies, subject_psd=subject_psd, channels=channels,
             subjects=subjects, subject_seconds=subject_seconds,
             subject_stimulus_psd=subject_stimulus_psd, n_files=len(paths))
    print(f"written: {cache.name}")

UV2 = 1e12                                            # V^2/Hz -> uV^2/Hz
subject_mean = subject_psd.mean(-1) * UV2            # (S, F): each participant's channel mean

# --- outliers --------------------------------------------------------------------
# A few recordings carry far more power than the rest -- two of them ~190x the median
# participant's -- and dominate any mean. True leaves out every participant whose total power
# over OUTLIER_BAND (mains bins excluded) has a modified z-score above OUTLIER_Z: the
# distance of its log power from the median, in units of the median absolute deviation,
# scaled by 0.6745 (Iglewicz & Hoaglin, whose recommended cutoff 3.5 is the default).
EXCLUDE_OUTLIERS = True
OUTLIER_BAND = (0.2, 250.0)
OUTLIER_Z = 3.5

harmonics = np.arange(50.0, frequencies[-1] + 50.0, 50.0)
mains = (np.abs(frequencies[:, None] - harmonics) <= 2.0).any(1)
outlier_bins = (frequencies >= OUTLIER_BAND[0]) & (frequencies <= OUTLIER_BAND[1]) & ~mains
log_power = np.log10(subject_mean[:, outlier_bins].sum(1))
deviation = log_power - np.median(log_power)
modified_z = 0.6745 * deviation / np.median(np.abs(deviation))
outlier = modified_z > OUTLIER_Z
print(f"outliers (modified z > {OUTLIER_Z:g} over {OUTLIER_BAND[0]:g}-{OUTLIER_BAND[1]:g} Hz): "
      + (", ".join(f"{subjects[i]} (z {modified_z[i]:.1f}, {10 ** deviation[i]:.0f}x the median power)"
                   for i in np.flatnonzero(outlier)) or "none")
      + ("  -> excluded" if EXCLUDE_OUTLIERS else "  -> kept (EXCLUDE_OUTLIERS = False)"))
if EXCLUDE_OUTLIERS:
    subject_mean, subject_seconds = subject_mean[~outlier], subject_seconds[~outlier]
    subject_psd = subject_psd[~outlier]
    subject_stimulus_psd = subject_stimulus_psd[~outlier]
    subjects = [name for name, out in zip(subjects, outlier) if not out]

# --- channel outliers --------------------------------------------------------------
# The same rule within each channel: a participant's log power over OUTLIER_BAND on that channel
# against the other participants' on the same channel. A bad electrode in one recording (sub-065's
# Iz carries ~750x the median power) is not caught by the participant rule, since one channel
# barely moves a 64-channel mean. True drops such participant-channel pairs from that channel's
# averages and from that participant's channel mean; the participant's other channels stay.
EXCLUDE_CHANNEL_OUTLIERS = True

channel_log_power = np.log10(subject_psd[:, outlier_bins, :].sum(1) * UV2)      # (S, C)
channel_deviation = channel_log_power - np.median(channel_log_power, axis=0)
channel_z = 0.6745 * channel_deviation / np.median(np.abs(channel_deviation), axis=0)
channel_outlier = channel_z > OUTLIER_Z                                         # (S, C)
worst = np.argsort(channel_z, axis=None)[::-1][:6]
print(f"channel outliers (modified z > {OUTLIER_Z:g} within a channel): "
      f"{int(channel_outlier.sum())} of {channel_outlier.size} participant-channel pairs, "
      f"in {int(channel_outlier.any(1).sum())} participants; largest "
      + ", ".join(f"{subjects[i]} {channels[j]} ({10 ** channel_deviation[i, j]:.0f}x)"
                  for i, j in zip(*np.unravel_index(worst, channel_z.shape)))
      + ("  -> excluded" if EXCLUDE_CHANNEL_OUTLIERS else "  -> kept (EXCLUDE_CHANNEL_OUTLIERS = False)"))
if EXCLUDE_CHANNEL_OUTLIERS:
    subject_psd = np.where(channel_outlier[:, None, :], np.nan, subject_psd)

# The exclusions, summarised for the methods.
n_total = len(outlier)
per_participant = channel_outlier.sum(1)
per_channel = channel_outlier.sum(0)
print(f"\nexclusion summary: {int(outlier.sum()) if EXCLUDE_OUTLIERS else 0} of {n_total} participants "
      f"excluded whole; of the remaining {len(subjects)} x {len(channels)} = {channel_outlier.size} "
      f"participant-channel pairs, {int(channel_outlier.sum()) if EXCLUDE_CHANNEL_OUTLIERS else 0} "
      f"({100 * channel_outlier.mean():.1f}%) excluded")
print(f"  participants with any channel excluded: {int((per_participant > 0).sum())}; channels "
      f"excluded per such participant: median {int(np.median(per_participant[per_participant > 0]))}, "
      f"max {int(per_participant.max())} ({subjects[int(np.argmax(per_participant))]})")
counts = np.bincount(per_participant)
print("  participants by number of channels excluded: "
      + ", ".join(f"{k}: {counts[k]}" for k in range(len(counts)) if counts[k]))
order = np.argsort(per_channel)[::-1]
print(f"  channels with any participant excluded: {int((per_channel > 0).sum())} of {len(channels)}; "
      f"most often: " + ", ".join(f"{channels[j]} {per_channel[j]}" for j in order[:8]))
# Each participant's channel mean over the channels they keep.
subject_mean = np.nanmean(subject_psd, axis=-1) * UV2
# Participants weighted by their analysed listening time, everywhere.
subject_weight = subject_seconds / subject_seconds.sum()


def weighted_quantile(values, weights, quantile):
    """Per column of `values` (S, F): the weighted `quantile`, by the midpoint rule on the
    cumulative weight, which reduces to the ordinary median for equal weights and odd S."""
    order = np.argsort(values, axis=0)
    ordered = np.take_along_axis(values, order, axis=0)
    cumulative = np.cumsum(weights[order], axis=0) - 0.5 * weights[order]
    return np.array([np.interp(quantile, cumulative[:, j], ordered[:, j])
                     for j in range(values.shape[1])])


grand = subject_weight @ subject_mean                # duration-weighted mean
median = weighted_quantile(subject_mean, subject_weight, 0.5)
spread = np.array([weighted_quantile(subject_mean, subject_weight, q) for q in (0.1, 0.9)])
print(f"listening time per participant {subject_seconds.min() / 3600:.2f}-"
      f"{subject_seconds.max() / 3600:.2f} h (median {np.median(subject_seconds) / 3600:.2f} h)")
print(f"{len(subjects)} participants in the averages, {len(channels)} channels, {len(frequencies)} bins "
      f"at {frequencies[1] - frequencies[0]:g} Hz")

## Figure

Log-log, from the first non-DC bin to the Nyquist frequency. The mains line and its
harmonics are left in, as the spectra were estimated, so they show as narrow spikes.

In [ ]:
# ===================== figure options ========================================
# Matched to notebook 2's grand-average coherence figure: its 6.27 in width, and a height that
# gives this panel the same plotting area as that figure's spectrum panel (about 4.9 x 2.1 in
# under the default subplot margins), with the same frequency axis -- log, 0.09-750 Hz, ticks
# at 0.1, 1, 10 and 100 -- and decade ticks written as powers of ten.
PSD_FIGSIZE = (6.27, 2.75)
PSD_XLIM = (0.09, 750.0)
PSD_XTICKS = [0.1, 1, 10, 100]
# Figure 2's five coherence bands: the band powers printed below, and optional shading.
COHERENCE_BANDS = [(0.2, 2.0), (4.0, 15.0), (30.0, 60.0), (80.0, 120.0), (140.0, 250.0)]
SHADE_COHERENCE_BANDS = False   # True shades them on both figures, for orientation
BAND_SHADE_ALPHA = 0.35
# =============================================================================

# The data start at the first non-DC bin; the axis runs a little wider, as figure 2's does.
shown = (frequencies > 0) & (frequencies <= PSD_XLIM[1])


def style_psd_axes(ax, low, high):
    """Figure 2's frequency axis, and decade y ticks spanning `low`-`high`."""
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(*PSD_XLIM)
    ax.set_xticks(PSD_XTICKS, [f"{tick:g}" for tick in PSD_XTICKS], fontsize=TICK_FONTSIZE)
    # Minor log ticks on the frequency axis, as figure 2 has them, unlabelled; none on the
    # power axis, where eight decades of them crowd the spine.
    ax.xaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10), numticks=100))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.yaxis.set_minor_locator(NullLocator())
    decades = np.arange(np.floor(np.log10(low)), np.ceil(np.log10(high)) + 1).astype(int)
    ax.set_yticks([10.0 ** d for d in decades], [f"$10^{{{d}}}$" for d in decades],
                  fontsize=TICK_FONTSIZE)
    ax.set_ylim(10.0 ** decades[0], 10.0 ** decades[-1])
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel(r"PSD ($\mu$V$^2$/Hz)")
    bare(ax)
fig, ax = plt.subplots(figsize=PSD_FIGSIZE)
for low, high in (COHERENCE_BANDS if SHADE_COHERENCE_BANDS else []):
    ax.axvspan(low, high, color=HAIRLINE, alpha=BAND_SHADE_ALPHA, lw=0, zorder=0)
ax.fill_between(frequencies[shown], spread[0][shown], spread[1][shown], color=NULL_FILL,
                alpha=0.6, lw=0, zorder=1, label="Participants, 10th-90th percentile (weighted)")
ax.plot(frequencies[shown], median[shown], color=MUTED, lw=0.8, ls="--", zorder=2,
        label="Median (weighted)")
ax.plot(frequencies[shown], grand[shown], color=INK, lw=1.0, zorder=3, label="Mean (weighted)")
style_psd_axes(ax, spread[0][shown].min(), max(spread[1][shown].max(), grand[shown].max()))
ax.legend(fontsize=SMALL_FONTSIZE, frameon=False, loc="lower left")
ax.set_title("Grand average EEG power spectral density")

stem = (f"eeg_psd_{REFERENCE}" + ("_outliers_excluded" if EXCLUDE_OUTLIERS else "")
        + ("_channel_outliers_excluded" if EXCLUDE_CHANNEL_OUTLIERS else ""))
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

# Numbers for the caption or text. Mains bins (+/-2 Hz about every 50 Hz harmonic) are left
# out of every fit and band power, as the other notebooks leave them out of their fits.
alpha = (frequencies >= 6) & (frequencies <= 14)
fit = (frequencies >= 2) & (frequencies <= 40) & ~alpha & ~mains
for label, spectrum in [("grand average", grand), ("median participant", median)]:
    slope, intercept = np.polyfit(np.log10(frequencies[fit]), np.log10(spectrum[fit]), 1)
    # The alpha peak as the largest excess over the 1/f line, not the largest power: on a
    # falling spectrum the raw maximum in any window is just its lower edge.
    excess = np.log10(spectrum) - (slope * np.log10(np.maximum(frequencies, 1e-9)) + intercept)
    peak = frequencies[alpha][np.argmax(excess[alpha])]
    print(f"{label}: 1/f slope over 2-40 Hz (alpha and mains excluded) {slope:.2f}; "
          f"alpha peak {peak:.1f} Hz, {10 * excess[alpha].max():.1f} dB above the 1/f line")
for f in (1, 10, 50, 100, 200):
    i = np.argmin(np.abs(frequencies - f))
    print(f"  {f:4d} Hz: {grand[i]:.3g} uV^2/Hz")
df = frequencies[1] - frequencies[0]
for label, spectrum in [("grand average", grand), ("median participant", median)]:
    print(f"band RMS, mains excluded ({label}): " + ", ".join(
        f"{low:g}-{high:g} Hz {np.sqrt(spectrum[(frequencies >= low) & (frequencies <= high) & ~mains].sum() * df):.2f} uV"
        for low, high in COHERENCE_BANDS))

## Figure -- every channel

Each channel's own duration-weighted average over the participants in the averages, with the
same outlier exclusion as above. Channels are coloured by their position on the scalp, by
the scheme notebook 6 uses (nearby sensors get nearby colours), and the key shows which
colour is where. The bold black line is the mean over the channels.

In [ ]:
from mne import pick_info
from mne.io import read_raw_fif
from mne.viz.evoked import _get_pos_outlines, _rgb
from mne.viz.topomap import _draw_outlines

# ===================== channel figure options ================================
CHANNELS_FIGSIZE = PSD_FIGSIZE                  # the same size and axes as the figure above
CHANNELS_KEY_RECT = [0.0, 0.02, 0.16, 0.42]     # sensor key, in axes fraction
# =============================================================================

TEMPLATE_EEG = (Path(CONFIG["eeg_dir"])
                / "sub-001_-_ses-001_-_trial-001_-_cond-natural_-_audio01-audiobook_5_1_-_eeg.fif")
RAW_INFO = read_raw_fif(TEMPLATE_EEG, preload=False, verbose="ERROR").pick("eeg").info
INFO = pick_info(RAW_INFO, [RAW_INFO.ch_names.index(name) for name in channels])
# Position to colour, as notebook 6: nearby channels get nearby colours.
CHANNEL_COLOURS = _rgb(*np.array([channel["loc"][:3] for channel in INFO["chs"]]).T)


def plot_sensor_key(ax):
    """The head that says which colour is where."""
    positions, outlines = _get_pos_outlines(INFO, np.arange(len(INFO.ch_names)),
                                            sphere=(0, 0, 0, 0.095))
    _draw_outlines(ax, outlines)
    ax.scatter(*positions.T, c=CHANNEL_COLOURS, s=5, linewidths=0)
    ax.set_aspect("equal")
    ax.set_axis_off()


# Each channel's duration-weighted average over the participants it keeps, and their mean.
# With channel outliers dropped, a channel's weights are renormalised over whoever is left on it,
# so this mean of channels differs slightly from the figure above's mean of participants.
kept = ~np.isnan(subject_psd[:, 0, :])                                # (S, C)
channel_weight = subject_seconds[:, None] * kept
channel_psd = (np.nansum(subject_psd * channel_weight[:, None, :], axis=0)
               / channel_weight.sum(0) * UV2)                         # (F, C)
channel_mean = channel_psd.mean(-1)

fig, ax = plt.subplots(figsize=CHANNELS_FIGSIZE)
for low, high in (COHERENCE_BANDS if SHADE_COHERENCE_BANDS else []):
    ax.axvspan(low, high, color=HAIRLINE, alpha=BAND_SHADE_ALPHA, lw=0, zorder=0)
for index in range(channel_psd.shape[1]):
    ax.plot(frequencies[shown], channel_psd[shown, index], color=CHANNEL_COLOURS[index],
            lw=0.5, alpha=0.7, zorder=2)
ax.plot(frequencies[shown], channel_mean[shown], color=INK, lw=1.8, zorder=3)
style_psd_axes(ax, channel_psd[shown].min(), channel_psd[shown].max())
plot_sensor_key(ax.inset_axes(CHANNELS_KEY_RECT))
ax.set_title("Grand average EEG power spectral density")

stem = (f"eeg_psd_channels_{REFERENCE}" + ("_outliers_excluded" if EXCLUDE_OUTLIERS else "")
        + ("_channel_outliers_excluded" if EXCLUDE_CHANNEL_OUTLIERS else ""))
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

for low, high in [(0.2, 2.0), (30.0, 250.0)]:
    rows = (frequencies >= low) & (frequencies <= high) & ~mains
    rms = np.sqrt(channel_psd[rows].sum(0) * (frequencies[1] - frequencies[0]))
    order = np.argsort(rms)[::-1]
    print(f"{low:g}-{high:g} Hz RMS per channel: {rms.min():.2f}-{rms.max():.2f} uV; largest "
          + ", ".join(f"{channels[i]} {rms[i]:.2f}" for i in order[:4]) + "; smallest "
          + ", ".join(f"{channels[i]} {rms[i]:.2f}" for i in order[-3:]))

## For fun -- the stimulus PSD over the EEG PSD

The speech envelope's PSD, in physical units (Pa$^{1.2}$/Hz, from each stimulus's calibrated
62 dB(A) level, as notebook 6 converts it), pooled with the same weights and participants as
the EEG, divided by each channel's EEG PSD. Colours and the black channel mean as above.

It is the regressor-to-background power ratio the transfer function is estimated against.
Since $|H|^2 = |C_{xy}|^2 S_{yy} / S_{xx}$, this ratio is $|C_{xy}|^2 / |H|^2$: where it is
large, a given coherence corresponds to a small gain.

In [ ]:
# The envelope PSD, weighted over the participants kept as the EEG is (per channel, since channel
# outliers renormalise each channel's weights).
stimulus_psd = np.tensordot(subject_seconds / subject_seconds.sum(), subject_stimulus_psd, axes=1)
ratio = np.nansum(subject_stimulus_psd[:, :, None] * channel_weight[:, None, :], axis=0) \
    / channel_weight.sum(0) / channel_psd                        # Pa^1.2 / uV^2, (F, C)
ratio_mean = ratio.mean(-1)

fig, ax = plt.subplots(figsize=CHANNELS_FIGSIZE)
for index in range(ratio.shape[1]):
    ax.plot(frequencies[shown], ratio[shown, index], color=CHANNEL_COLOURS[index],
            lw=0.5, alpha=0.7, zorder=2)
ax.plot(frequencies[shown], ratio_mean[shown], color=INK, lw=1.8, zorder=3)
# Limits from the non-mains bins: the EEG's line noise makes the ratio dip by orders of
# magnitude at each harmonic, which would otherwise set the whole axis.
style_psd_axes(ax, ratio[shown & ~mains].min(), ratio[shown & ~mains].max())
ax.set_ylabel(r"Stimulus / EEG PSD (Pa$^{1.2}$/$\mu$V$^2$)")
ax.set_title("Stimulus PSD over EEG PSD")
plot_sensor_key(ax.inset_axes(CHANNELS_KEY_RECT))

stem = (f"stimulus_to_eeg_psd_{REFERENCE}" + ("_outliers_excluded" if EXCLUDE_OUTLIERS else "")
        + ("_channel_outliers_excluded" if EXCLUDE_CHANNEL_OUTLIERS else ""))
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

for f in (0.5, 4, 10, 40, 100, 200):
    i = np.argmin(np.abs(frequencies - f))
    print(f"{f:5g} Hz: envelope {stimulus_psd[i]:.3g} Pa^1.2/Hz, EEG {channel_mean[i]:.3g} uV^2/Hz, "
          f"ratio (channel mean) {ratio_mean[i]:.3g} Pa^1.2/uV^2")

## The transfer function

$|H(f)|$ per channel, in µV per Pa$^{0.6}$, with no EEG normalisation. For each trial, the
cross-spectrum is divided by the envelope's PSD and converted to Pa$^{0.6}$ with the same
per-stimulus factors as above. The result is averaged as the PSDs are: listening-time-weighted
within each participant, then across participants with the same participant and channel
exclusions. The complex $H$ is averaged before its magnitude is taken, as notebook 6 does, so
what survives is the part of the response consistent across trials and participants. Colours
and the black channel mean (of the magnitudes) as above.

At each frequency this is the gain from envelope to EEG. Together with the two figures above,
$|C_{xy}|^2 = |H|^2 S_{xx} / S_{yy}$.

In [ ]:
import hashlib

from trfs.transfer_function import rereference_transfer_function


def pool_transfer_function():
    """Complex H per channel, (F, C) in V per Pa^0.6: duration-weighted within participants,
    then across the participants kept, each channel over the participants it keeps."""
    keep_subject = set(subjects)
    channel_kept = {name: ~np.isnan(subject_psd[i, 0, :]) for i, name in enumerate(subjects)}
    seconds = dict(zip(subjects, subject_seconds))
    sums, weights = {}, {}
    for count, path in enumerate(paths, 1):
        trial = load_trial(path, FEATURE)
        if trial.subject not in keep_subject:
            continue
        cross, names = rereference_transfer_function(trial.cross_spectrum, trial.channel_names,
                                                     REFERENCE)
        cross = cross[:, [list(names).index(name) for name in channels]]
        stimulus = trial.stimulus
        # Digital units -> Pa^0.6: the envelope scales as k^0.6, its PSD as k^1.2, H as k^-0.6.
        H = np.divide(cross, trial.stimulus_psd[:, None], out=np.zeros_like(cross),
                      where=trial.stimulus_psd[:, None] > 0) / np.sqrt(ENVELOPE_POWER_FACTOR[stimulus])
        weight = trial.analysed_duration
        sums[trial.subject] = H * weight + sums.get(trial.subject, 0.0)
        weights[trial.subject] = weights.get(trial.subject, 0.0) + weight
        if count % 100 == 0 or count == len(paths):
            print(f"  {count}/{len(paths)} trials", flush=True)
    total = np.zeros(sums[subjects[0]].shape, complex)
    total_weight = np.zeros(len(channels))
    for name in subjects:
        w = seconds[name] * channel_kept[name]                          # (C,)
        total += sums[name] / weights[name] * w[None, :]
        total_weight += w
    return total / total_weight


tf_key = json.dumps([REFERENCE, WINDOW_SECONDS, OVERLAP, len(paths), subjects,
                     EXCLUDE_CHANNEL_OUTLIERS, OUTLIER_Z, OUTLIER_BAND], sort_keys=True)
tf_cache = CACHE_DIR / f"eeg_transfer_function_{hashlib.sha1(tf_key.encode()).hexdigest()[:12]}.npz"
if tf_cache.exists():
    with np.load(tf_cache) as saved:
        transfer_function = saved["transfer_function"]
    print(f"cached: {tf_cache.name}")
else:
    transfer_function = pool_transfer_function()
    np.savez(tf_cache, transfer_function=transfer_function)
    print(f"written: {tf_cache.name}")

gain = np.abs(transfer_function) * 1e6                                  # uV per Pa^0.6
gain_mean = gain.mean(-1)
nonzero = shown & ~mains

fig, ax = plt.subplots(figsize=CHANNELS_FIGSIZE)
for index in range(gain.shape[1]):
    ax.plot(frequencies[shown], gain[shown, index], color=CHANNEL_COLOURS[index],
            lw=0.5, alpha=0.7, zorder=2)
ax.plot(frequencies[shown], gain_mean[shown], color=INK, lw=1.8, zorder=3)
style_psd_axes(ax, gain[nonzero].min(), gain[nonzero].max())
ax.set_ylabel(r"$|H|$ ($\mu$V Pa$^{-0.6}$)")
ax.set_title("Transfer function magnitude")
plot_sensor_key(ax.inset_axes(CHANNELS_KEY_RECT))

stem = (f"transfer_function_magnitude_{REFERENCE}" + ("_outliers_excluded" if EXCLUDE_OUTLIERS else "")
        + ("_channel_outliers_excluded" if EXCLUDE_CHANNEL_OUTLIERS else ""))
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

for f in (0.5, 4, 10, 40, 100, 200):
    i = np.argmin(np.abs(frequencies - f))
    order = np.argsort(gain[i])[::-1]
    print(f"{f:5g} Hz: |H| channel mean {gain_mean[i]:.3g} uV/Pa^0.6 "
          f"(largest {channels[order[0]]} {gain[i, order[0]]:.3g})")

## Four panels: stimulus, EEG, transfer function, and their combination

* **(a)** The envelope's PSD, $S_{xx}$, in Pa$^{1.2}$/Hz: one line, since the stimulus has no
  channels. `SHOW_INDIVIDUAL_STIMULI` adds each stimulus's own spectrum faintly underneath (off). In grey, the same with the pauses removed: notebook 1's silence detection (a
  per-stimulus Otsu cut on the smoothed envelope level, silences under 150 ms kept as
  speech-internal, segments padded by 25 ms and spliced with 20 ms crossfades), estimated the
  same way (5 s Hann windows, 80% overlap), converted to Pa$^{1.2}$/Hz with the same factors and
  weighted over stimuli exactly as $S_{xx}$ is. The splicing shapes the lowest frequencies, so
  the grey line's low end is approximate.
* **(b)** The EEG PSD, $S_{yy}$, per channel (the main figure above).
* **(c)** The squared transfer function magnitude $|H|^2$, per channel, so that (d) is (c)
  times (a) over (b).
* **(d)** $|H|^2 S_{xx} / S_{yy}$, per channel: the magnitude-squared coherence the pooled fit implies.
  $H$ is averaged over trials and participants before it is squared, so this counts only the part
  of the coupling that is consistent across them, as figure 2's complex-averaged coherence does.
  It is dimensionless.

All four use the weights, exclusions and units above, and figure 2's frequency axis. The
channel panels share the colouring, with the channel mean in black, and each carries the key.
(c) and (d) are smoothed for display over 1/6 octave (`OVERVIEW_SMOOTHING_OCTAVES`, mains bins
left out); the printed values are not.

In [ ]:
# --- the envelope without its pauses, for panel (a) --------------------------------
from scipy.signal import welch

ENVELOPE_FS = 1024.0
FEATURES_DIR = Path(CONFIG["audio_features_dir"])


def otsu_threshold(values, lo=-60.0, hi=5.0, n_bins=256):
    """Cut a bimodal distribution at the valley between its two modes (notebook 1)."""
    counts, _ = np.histogram(np.clip(values, lo, hi), bins=n_bins, range=(lo, hi))
    edges = np.linspace(lo, hi, n_bins + 1)
    centres = 0.5 * (edges[1:] + edges[:-1])
    p = counts / counts.sum()
    w0 = np.cumsum(p)
    w1 = 1.0 - w0
    cumulative_mean = np.cumsum(p * centres)
    valid = (w0 > 1e-6) & (w1 > 1e-6)
    variance = np.zeros(n_bins)
    variance[valid] = ((cumulative_mean[-1] * w0[valid] - cumulative_mean[valid]) ** 2
                       / (w0[valid] * w1[valid]))
    return float(centres[np.argmax(variance)])


def _runs(mask, value):
    flags = (mask == value).view(np.int8)
    edges = np.flatnonzero(np.diff(np.concatenate(([0], flags, [0]))))
    return edges.reshape(-1, 2)


def find_speech_segments(envelope, fs=ENVELOPE_FS, smooth_ms=50.0, min_silence_ms=150.0,
                         min_speech_ms=100.0, pad_ms=25.0):
    """Sample ranges of the envelope that are speech rather than silence (notebook 1)."""
    window = max(1, int(round(smooth_ms * 1e-3 * fs)))
    smoothed = np.convolve(np.abs(envelope), np.ones(window) / window, mode="same")
    reference = max(np.percentile(smoothed, 95), 1e-12)
    level_db = 20 * np.log10(np.maximum(smoothed, 1e-12) / reference)
    is_speech = level_db > otsu_threshold(level_db)
    for start, stop in _runs(is_speech, False):
        if stop - start < int(round(min_silence_ms * 1e-3 * fs)):
            is_speech[start:stop] = True
    for start, stop in _runs(is_speech, True):
        if stop - start < int(round(min_speech_ms * 1e-3 * fs)):
            is_speech[start:stop] = False
    pad = int(round(pad_ms * 1e-3 * fs))
    segments = []
    for start, stop in _runs(is_speech, True):
        start, stop = max(0, start - pad), min(envelope.size, stop + pad)
        if segments and start <= segments[-1][1]:
            segments[-1] = (segments[-1][0], max(segments[-1][1], stop))
        else:
            segments.append((start, stop))
    return segments


def concatenate_segments(envelope, segments, fs=ENVELOPE_FS, crossfade_ms=20.0):
    """Splice segments together, crossfading the joins (notebook 1)."""
    crossfade = int(round(crossfade_ms * 1e-3 * fs))
    pieces = [envelope[start:stop] for start, stop in segments if stop > start]
    out = pieces[0].copy()
    for piece in pieces[1:]:
        n = min(crossfade, out.size, piece.size)
        ramp = np.linspace(0.0, 1.0, n, endpoint=False)
        out[-n:] = out[-n:] * (1.0 - ramp) + piece[:n] * ramp
        out = np.concatenate([out, piece[n:]])
    return out


def envelope_psd(signal):
    """The estimator the trial files' S_xx uses: Welch, 5 s Hann windows, 80% overlap."""
    return welch(signal, fs=ENVELOPE_FS, nperseg=int(ENVELOPE_FS * WINDOW_SECONDS),
                 noverlap=int(ENVELOPE_FS * WINDOW_SECONDS * OVERLAP))[1]


# Which stimulus each kept participant's trials were on, and for how long: the weights S_xx carries.
stimulus_seconds = {}
for path in paths:
    with np.load(path) as saved:
        if str(saved["subject"]) in set(subjects):
            name = str(saved["stimulus"])
            stimulus_seconds[name] = stimulus_seconds.get(name, 0.0) + float(saved["analysed_duration"])

pause_cache = OUT_DIR / "figure_data" / f"stimulus_envelope_psd_pauses_{WINDOW_SECONDS:g}s.npz"
known = {}
if pause_cache.exists():
    with np.load(pause_cache) as saved:
        known = {str(n): (w, wo, r) for n, w, wo, r in
                 zip(saved["stimuli"], saved["with_pauses"], saved["without_pauses"], saved["removed"])}
if not set(stimulus_seconds) <= set(known):
    for name in sorted(set(stimulus_seconds) - set(known)):
        with np.load(FEATURES_DIR / f"{name}.npz") as features:
            envelope = np.clip(features["gammatone_envelope"].squeeze(), 0, None)
        trimmed = concatenate_segments(envelope, find_speech_segments(envelope))
        known[name] = (envelope_psd(envelope), envelope_psd(trimmed), 1 - trimmed.size / envelope.size)
    names = sorted(known)
    np.savez(pause_cache, stimuli=np.array(names), with_pauses=np.array([known[n][0] for n in names]),
             without_pauses=np.array([known[n][1] for n in names]),
             removed=np.array([known[n][2] for n in names]))

total_seconds = sum(stimulus_seconds.values())
pause_psd = {kind: sum(known[name][index] * ENVELOPE_POWER_FACTOR[name] * seconds
                       for name, seconds in stimulus_seconds.items()) / total_seconds
             for kind, index in [("with", 0), ("without", 1)]}
check = np.median(pause_psd["with"][1:] / stimulus_psd[1:])
# Each stimulus heard by the participants kept, in Pa^1.2/Hz, pauses kept: panel (a)'s faint lines.
individual_stimulus_psd = np.array([known[name][0] * ENVELOPE_POWER_FACTOR[name]
                                    for name in sorted(stimulus_seconds)])
removed = np.array([known[name][2] for name in stimulus_seconds])
print(f"pauses removed: {100 * removed.min():.0f}-{100 * removed.max():.0f}% of each stimulus "
      f"(median {100 * np.median(removed):.0f}%); this estimate with pauses / S_xx: median {check:.3f}")

# ===================== overview figure options ===============================
OVERVIEW_FIGSIZE = (6.27, 4.6)
OVERVIEW_KEY_RECT = [0.0, 0.02, 0.24, 0.42]     # sensor key in (b), in axes fraction
OVERVIEW_LOWER_PERCENTILE = 1.0                 # y floor; traces below it are clipped
OVERVIEW_YFLOOR = {"c)": 1e-4}                  # per-panel fixed y floor, overriding the above
SHOW_INDIVIDUAL_STIMULI = False                 # True draws every stimulus's own PSD faintly in (a)
# Display-only smoothing of (c) and (d): each trace averaged over a window this many octaves
# wide, centred on each bin, with the mains bins left out of every average. Per-bin estimates of
# H are noisy above ~15 Hz; a fixed fraction of an octave smooths the high end without blurring
# the low end, where the window is narrower than one 0.2 Hz bin. 0 draws the raw estimates.
# Printed numbers are never smoothed.
OVERVIEW_SMOOTHING_OCTAVES = 1 / 6
SMOOTHED_PANELS = ("c)", "d)")
KEY_PANELS = ("b)", "c)", "d)")


def octave_smooth(values, octaves):
    """`values` (F, C) averaged over +/- octaves/2 about each bin, mains bins excluded."""
    if not octaves:
        return values
    valid = ~mains & (frequencies > 0)
    cumulative = np.vstack([np.zeros((1, values.shape[1])),
                            np.cumsum(np.where(valid[:, None], values, 0.0), axis=0)])
    counts = np.concatenate([[0], np.cumsum(valid)])
    low = np.searchsorted(frequencies, frequencies * 2 ** (-octaves / 2), side="left")
    high = np.searchsorted(frequencies, frequencies * 2 ** (octaves / 2), side="right")
    n = (counts[high] - counts[low])[:, None]
    smoothed = (cumulative[high] - cumulative[low]) / np.maximum(n, 1)
    return np.where(n > 0, smoothed, np.nan)
# =============================================================================

# The envelope PSD per channel's own participants, so (d) divides like with like.
stimulus_by_channel = (np.nansum(subject_stimulus_psd[:, :, None] * channel_weight[:, None, :], axis=0)
                       / channel_weight.sum(0))                              # (F, C), Pa^1.2/Hz
implied_coherence = gain ** 2 * stimulus_by_channel / channel_psd           # (uV/Pa^0.6)^2 Pa^1.2/uV^2
# By channel, the four quantities; (a) is a single trace.
PANELS = [
    ("a)", "Mean stimulus PSD", r"$S_{xx}$ (Pa$^{1.2}$/Hz)", stimulus_psd[:, None], False),
    ("b)", "Mean EEG PSD", r"$S_{yy}$ ($\mu$V$^2$/Hz)", channel_psd, True),
    ("c)", "Magnitude of \nmean transfer function", r"$|H|^2$ ($\mu$V/Pa$^{0.6}$)$^2$", gain ** 2, True),
    ("d)", "Implied magnitude-\nsquared coherence", r"$|H|^2 S_{xx} / S_{yy}$", implied_coherence, True),
]

fig, axes = plt.subplots(2, 2, figsize=OVERVIEW_FIGSIZE,
                         gridspec_kw=dict(hspace=0.6, wspace=0.42))
for ax, (label, title, ylabel, values, by_channel) in zip(axes.flat, PANELS):
    if label in SMOOTHED_PANELS:
        values = octave_smooth(values, OVERVIEW_SMOOTHING_OCTAVES)
    if by_channel:
        for index in range(values.shape[1]):
            ax.plot(frequencies[shown], values[shown, index], color=CHANNEL_COLOURS[index],
                    lw=0.4, alpha=0.7, zorder=2)
    if label == "a)":
        # Every stimulus's own spectrum (pauses kept), faint, under the two averages.
        for psd in (individual_stimulus_psd if SHOW_INDIVIDUAL_STIMULI else []):
            ax.plot(frequencies[shown], psd[shown], color=NULL_FILL, lw=0.4, alpha=0.5, zorder=1)
        ax.plot(frequencies[shown], pause_psd["without"][shown], color=MUTED, lw=1.0, ls="--",
                zorder=2, label="Pauses removed")
    ax.plot(frequencies[shown], np.nanmean(values[shown], -1), color=INK,
            lw=1.5 if by_channel else 1.0, zorder=3,
            label=r"$S_{xx}$ (whole stimulus)" if label == "a)" else None)
    # The lower limit at the 1st percentile of the channel traces: per-bin estimates of |H| are
    # noisy, and a handful of near-zero bins would otherwise stretch the axis by decades.
    finite = values[nonzero][np.isfinite(values[nonzero])]
    if label == "a)":
        finite = np.concatenate([finite, pause_psd["without"][nonzero]]
                                + ([individual_stimulus_psd[:, nonzero].ravel()]
                                   if SHOW_INDIVIDUAL_STIMULI else []))
    style_psd_axes(ax, OVERVIEW_YFLOOR.get(label, np.percentile(finite, OVERVIEW_LOWER_PERCENTILE)),
                   finite.max())
    if label == "a)":
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles[::-1], labels[::-1], fontsize=SMALL_FONTSIZE, frameon=False,
                  loc="lower left")                                  # S_xx first
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=DEFAULT_FONTSIZE)
    ax.text(-0.3, 1.04, label, transform=ax.transAxes, fontsize=PANEL_LABEL_FONTSIZE,
            fontweight="bold", va="bottom", ha="left")
for ax, (label, *_ ) in zip(axes.flat, PANELS):
    if label in KEY_PANELS:
        plot_sensor_key(ax.inset_axes(OVERVIEW_KEY_RECT))

stem = (f"spectra_overview_{REFERENCE}" + ("_outliers_excluded" if EXCLUDE_OUTLIERS else "")
        + ("_channel_outliers_excluded" if EXCLUDE_CHANNEL_OUTLIERS else ""))
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

for f in (0.6, 4, 10, 40, 100, 200):
    i = np.argmin(np.abs(frequencies - f))
    print(f"{f:5g} Hz: |H|^2 Sxx/Syy channel mean {implied_coherence[i].mean():.2e} "
          f"(sqrt {np.sqrt(implied_coherence[i].mean()):.3f}), largest "
          f"{channels[int(np.argmax(implied_coherence[i]))]} {implied_coherence[i].max():.2e}")

## Panel (d) against figure 2

Figure 2 draws the grand-average coherence as an RMS magnitude over the channels:
$\sqrt{\mathrm{mean}_k |\langle C_{xy_k}\rangle|^2}$, from notebook 2's cache. Panel (d) is a
magnitude-squared coherence per channel, so its counterpart is the square root of its mean over
channels, $\sqrt{\mathrm{mean}_k |H_k|^2 S_{xx} / S_{y_k y_k}}$. Both are drawn here, unsmoothed,
on figure 2's axes, with figure 2's null range in grey.

They are not the same estimator, so they need not agree exactly:

* **Figure 2** averages each trial's complex coherence: the cross-spectrum normalised by that
  trial's own EEG and envelope powers. Every participant is included.
* **Panel (d)** averages the cross-spectrum divided by the envelope power, and divides by the
  pooled EEG power only at the end. The outlier participants and channels are excluded.

The weighting is the same in both: every trial counts by its duration. Figure 2 pools trials
within stimuli and then stimuli by listening time, and this notebook pools them within
participants and then participants by listening time, which is the same sum.

What differs is where the EEG normalisation enters: per trial in figure 2, once at the end in
panel (d). Dividing by the pooled EEG power lets a few high-power trials enlarge the
denominator for all of them. That makes panel (d) lower, by 10-20% with the outliers excluded
and by up to 30% without them. The printout gives the ratio band by band.

In [ ]:
# ===================== comparison figure options =============================
COMPARISON_FIGSIZE = PSD_FIGSIZE
COMPARISON_COLOUR = "#c8442e"                    # notebook 2's accent, for the implied trace
NULL_COLOUR = "#b9b9b4"                          # notebook 2's NULL_FILL
# =============================================================================

grand_cache = OUT_DIR / "figure_data" / REFERENCE / "grand_average_coherence.npz"
if not grand_cache.exists():
    raise FileNotFoundError(f"{grand_cache} is written by notebook 2; run it first")
with np.load(grand_cache) as saved:
    assert np.allclose(saved["frequencies"], frequencies), "frequency grids differ"
    figure2 = np.sqrt((np.abs(saved["coherence"]) ** 2).mean(-1))        # (F,)
    figure2_null = (saved["null_spectra"].min(0), saved["null_spectra"].max(0))
implied_rms = np.sqrt(np.nanmean(implied_coherence, axis=-1))          # (F,)

fig, ax = plt.subplots(figsize=COMPARISON_FIGSIZE)
ax.fill_between(frequencies[shown], figure2_null[0][shown], figure2_null[1][shown],
                color=NULL_COLOUR, alpha=0.85, lw=0, zorder=1, label="Figure 2 null range")
# ax.plot(frequencies[shown], figure2[shown], color=INK, lw=0.8, zorder=3,
#         label=r"Figure 2: RMS of $|\langle C_{xy}\rangle|$")
ax.plot(frequencies[shown], implied_rms[shown], color=COMPARISON_COLOUR, lw=0.8, zorder=2,
        label=r"Panel (d): RMS of $\sqrt{|H|^2 S_{xx}/S_{yy}}$")
style_psd_axes(ax, figure2_null[0][nonzero].min(), max(figure2[nonzero].max(), implied_rms[nonzero].max()))
ax.set_ylabel("RMS coherence magnitude")
ax.set_title("Coherence: grand average and implied by the transfer function")
ax.legend(fontsize=SMALL_FONTSIZE, frameon=False, loc="lower left")

stem = (f"implied_vs_grand_average_coherence_{REFERENCE}"
        + ("_outliers_excluded" if EXCLUDE_OUTLIERS else "")
        + ("_channel_outliers_excluded" if EXCLUDE_CHANNEL_OUTLIERS else ""))
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

df = frequencies[1] - frequencies[0]
print(f"{'band':>14s} {'figure 2':>10s} {'panel (d)':>10s} {'ratio':>7s}   (band RMS of each trace, mains excluded)")
for low, high in COHERENCE_BANDS:
    rows = (frequencies >= low) & (frequencies <= high) & ~mains
    a, b = np.sqrt((figure2[rows] ** 2).mean()), np.sqrt((implied_rms[rows] ** 2).mean())
    print(f"{f'{low:g}-{high:g} Hz':>14s} {a:10.4f} {b:10.4f} {b / a:7.2f}")
null_median = np.median(figure2_null[0] + figure2_null[1]) / 2
print(f"figure 2 null range midpoint (median over bins) {null_median:.4f}")